# This notebook shows how observables disappear in simulation for certain model formats, namely regnets and stock-and-flow.
See https://github.com/gyorilab/mira/issues/312

### Load dependencies

In [1]:
import os
import json
import pyciemss

from mira.sources.amr import model_from_url, model_from_json_file
from mira.modeling.amr.regnet import template_model_to_regnet_json
from mira.modeling.amr.stockflow import template_model_to_stockflow_json

### Select models

In [2]:
# MODEL_PATH = "../../docs/source/"
MODEL_PATH = "https://raw.githubusercontent.com/DARPA-ASKEM/simulation-integration/main/data/models/"

petri_model = os.path.join(MODEL_PATH, "SEIRHD_base_model01_petrinet.json")
regnet_model = os.path.join(MODEL_PATH, "LV_rabbits_wolves_model03_regnet.json")
stockflow_model = os.path.join(MODEL_PATH, "SEIRHD_stockflow.json")

### Define function to load models from json or url

In [3]:
def load_model(path_to_model):
    if ".com" in path_to_model:
        model = model_from_url(path_to_model)
    else: 
        model = model_from_json_file(petri_model)  
    return model

### Set simulation parameters

In [4]:
start_time = 0.0
end_time = 50.0
logging_step_size = 10.0
num_samples = 10

### Load a petrinet model as a template model, and observe its observables

In [5]:
template_model = load_model(petri_model)
print(type(template_model))
display(template_model.observables)

<class 'mira.metamodel.template_model.TemplateModel'>


{'infected': Observable(name='infected', display_name='infected', description=None, identifiers={}, context={}, units=None, expression=I),
 'exposed': Observable(name='exposed', display_name='exposed', description=None, identifiers={}, context={}, units=None, expression=E),
 'hospitalized': Observable(name='hospitalized', display_name='hospitalized', description=None, identifiers={}, context={}, units=None, expression=H),
 'dead': Observable(name='dead', display_name='dead', description=None, identifiers={}, context={}, units=None, expression=D)}

### Simulate the model and see that observables appear in results

In [6]:
result1 = pyciemss.sample(template_model, end_time, logging_step_size, num_samples, start_time=start_time)
display(result1['data'].head())

,timepoint_id,sample_id,timepoint_unknown,persistent_beta_param,persistent_gamma_param,persistent_hosp_param,persistent_death_hosp_param,persistent_I0_param,D_state,E_state,H_state,I_state,R_state,S_state,infected_observable_state,exposed_observable_state,hospitalized_observable_state,dead_observable_state
0,0,0,10.0,0.746278,0.103353,0.137208,0.068417,5.059654,0.254634,286.353180,5.928075,196.198257,64.147430,19339498.0,196.198257,286.353180,5.928075,0.254634
1,1,0,20.0,0.746278,0.103353,0.137208,0.068417,5.059654,4.243609,3909.203857,82.340767,2679.530518,965.585388,19332402.0,2679.530518,3909.203857,82.340767,4.243609
2,2,0,30.0,0.746278,0.103353,0.137208,0.068417,5.059654,58.752499,53109.214844,1122.631714,36485.140625,13259.238281,19236008.0,36485.140625,53109.214844,1122.631714,58.752499
3,3,0,40.0,0.746278,0.103353,0.137208,0.068417,5.059654,792.590698,675026.937500,14927.069336,477851.687500,177503.234375,17993942.0,477851.687500,675026.937500,14927.069336,792.590698
4,0,1,10.0,0.184802,0.218280,0.165675,0.079303,11.161646,0.355189,19.540068,3.719439,24.031939,45.409927,19339948.0,24.031939,19.540068,3.719439,0.355189


### Now, load a regnet model, which does have observables in the AMR, and see that they do not propagate to the template version

In [11]:
template_model = load_model(regnet_model)
display(template_model.observables)

{}

### Sample the regnet model, and see that no observables appear in the sample results
There should be an `R_observable_state`, and a `W_observable_state`

In [8]:
result1 = pyciemss.sample(regnet_model, end_time, logging_step_size, num_samples, start_time=start_time)
display(result1['data'].head())

,timepoint_id,sample_id,timepoint_unknown,persistent_alpha_param,persistent_gamma_param,persistent_beta_param,persistent_delta_param,persistent_R0_param,persistent_W0_param,Rabbits_state,Wolves_state
0,0,0,10.0,0.529455,2.067616,0.542937,2.099762,1.140409,0.986760,0.921889,0.736373
1,1,0,20.0,0.529455,2.067616,0.542937,2.099762,1.140409,0.986760,0.898544,1.226475
2,2,0,30.0,0.529455,2.067616,0.542937,2.099762,1.140409,0.986760,1.138909,0.934864
3,3,0,40.0,0.529455,2.067616,0.542937,2.099762,1.140409,0.986760,0.903037,0.755140
4,0,1,10.0,0.493162,2.009615,0.466992,1.999727,1.194094,1.088343,0.875654,0.826333


### Repeat for a stock-and-flow model: no observables appear in the template model

In [13]:
template_model = load_model(stockflow_model)
display(template_model.observables)

{}

### Sample the stock-and-flow model, and see that no observables appear in the sample results
There should be an `infected_observable_state`, and a `hospitalized_observable_state`

In [10]:
result1 = pyciemss.sample(stockflow_model, end_time, logging_step_size, num_samples, start_time=start_time)
display(result1['data'].head())

,timepoint_id,sample_id,timepoint_unknown,persistent_p_cbeta_param,persistent_p_cdelta_param,persistent_p_cgamma_param,persistent_p_hosp_param,persistent_p_death_param,persistent_p_los_param,persistent_I0_param,D_state,E_state,H_state,I_state,R_state,S_state
0,0,0,10.0,0.344138,0.212048,0.223892,0.174971,0.054612,8.519632,9.175376,0.060044,9.432949,1.557364,7.416024,13.566922,968.966675
1,1,0,20.0,0.344138,0.212048,0.223892,0.174971,0.054612,8.519632,9.175376,0.196615,14.891405,2.754812,11.764626,33.368988,938.023743
2,2,0,30.0,0.344138,0.212048,0.223892,0.174971,0.054612,8.519632,9.175376,0.422317,22.093767,4.369541,17.854225,64.377052,891.883423
3,3,0,40.0,0.344138,0.212048,0.223892,0.174971,0.054612,8.519632,9.175376,0.767244,30.146898,6.462861,25.190367,109.999741,828.433167
4,0,1,10.0,0.257419,0.268549,0.125089,0.060707,0.033928,7.995835,3.657822,0.003874,3.610902,0.168791,4.897303,4.488566,987.830383
